# SignBridge Conversational Demo Seed

This notebook prepares a **conversational demo vocabulary** and a **demo model seed** for real-time ASL recognition.

We will:
- Load WLASL and ASL Citizen manifests (from Kaggle).
- Select a compact set of **pronouns, core verbs, and everyday nouns** that are good for short conversational sentences.
- Build a dedicated `label_mapping_demo.json` for this vocabulary.
- Prepare a training manifest and fine-tune a **demo model** starting from the 87.6% checkpoint.

> **Important:** This notebook is designed to run on **Kaggle**. Paths below assume Kaggle input mounts; you can adjust them as needed.

In [ ]:
# ---------- Cell 1: Imports & Paths (Kaggle-style) ----------

import os
import json
from collections import Counter

import pandas as pd

# In Kaggle, adjust these to match the actual dataset mounts.
# For local testing in this repo, you can override them to point to ./manifests.

# WLASL main manifest (JSON) - Kaggle path example
WLASL_JSON = os.environ.get(
    "WLASL_JSON_PATH",
    "/kaggle/input/wlasl2000-dataset/WLASL_v0.3.json",  # default Kaggle path
)

# ASL Citizen splits directory - Kaggle path example
CITIZEN_SPLITS_DIR = os.environ.get(
    "CITIZEN_SPLITS_DIR",
    "/kaggle/input/asl-citizen/ASL_Citizen/splits",  # default Kaggle path
)

# Output directory for manifests and label mappings inside Kaggle working dir
BASE_OUTPUT_DIR = os.environ.get("BASE_OUTPUT_DIR", "/kaggle/working/SignBridge_demo")
MANIFESTS_DIR = os.path.join(BASE_OUTPUT_DIR, "manifests")
os.makedirs(MANIFESTS_DIR, exist_ok=True)

print("WLASL_JSON =", WLASL_JSON)
print("CITIZEN_SPLITS_DIR =", CITIZEN_SPLITS_DIR)
print("MANIFESTS_DIR =", MANIFESTS_DIR)

# Demo-word selection philosophy (summary):
# - Start from all glosses in WLASL and (optionally) ASL Citizen.
# - Focus on a compact set (e.g., 30–50) of:
#   * Pronouns: I, YOU, WE, THEY, etc.
#   * Core verbs: WANT, NEED, LIKE, GO, COME, HELP, KNOW, WORK, etc.
#   * Everyday nouns: FAMILY, FRIEND, SCHOOL, HOME, TIME, YEAR, etc.
# - Prefer glosses that:
#   * Are reasonably frequent in WLASL/Citizen (enough videos to train).
#   * Are already present in the existing 100-label mapping when possible,
#     to maximize reuse of the 87.6% checkpoint.
# - Result: a small, conversational vocabulary for a robust real-time demo
#   model, separate from the full 100-label general model.